# 01 · Panorama 2022 → 2026

Primeira olhada nos dados processados. **Exploratório**: nada aqui é teste confirmatório;
o que virar afirmação vai antes para `docs/hipoteses.md` (hipótese A1, A8…).

**Para rodar:** os dados precisam estar em `data/processed/` (`just repro`, ou `just pull`
para baixar do R2).

**Vocabulário**

| termo | significado |
|---|---|
| `pt_2p` | fatia do PT entre os dois maiores: PT / (PT + adversário). Métrica principal entre pleitos, porque os candidatos menores mudam a cada eleição |
| adversário | Jair Bolsonaro (2022), Flávio Bolsonaro (2026) |
| pp | pontos percentuais: 52% → 48% é −4 pp |
| AMC | área mínima comparável: municípios agrupados para ter as mesmas fronteiras em 2022 e 2026 (hoje só Sorriso + Nova Ubiratã + Boa Esperança do Norte) |

Significado de cada coluna: `docs/dicionario_dados.md`.

## Preparação

Os gráficos são interativos (passe o mouse para ver o município e os valores). Cada um é
exibido como widget (`JupyterChart`) em **modo offline**: o JavaScript do Vega vem do
pacote `vl-convert`, sem depender de CDN. Os dados de cada gráfico são agregados antes,
para não embutir tabelas grandes na saída.

In [ ]:
import altair as alt
import polars as pl

from kingsrace import paths
from kingsrace.viz import tema  # noqa: F401  (registra e ativa o tema do projeto)
from kingsrace.viz.tema import SERIES, TINTA, escala_divergente

alt.renderers.enable("jupyter", offline=True)
alt.data_transformers.disable_max_rows()

P = paths.PROCESSED
municipios = pl.read_parquet(P / "municipios.parquet")  # cadastro: códigos, regiões, AMC
painel = pl.read_parquet(P / "painel_municipio.parquet")  # município × ano × turno
amc = pl.read_parquet(P / "painel_amc.parquet")  # idem, por AMC (para comparar anos)
votos = pl.read_parquet(P / "votos_municipio.parquet")  # município × candidato, com exterior

## 1. Resultado nacional

Votos e percentual dos válidos por candidato, **incluindo o exterior**. Candidatos com
menos de 1% omitidos. Serve de conferência: os números devem bater com os do TSE
(2022 1T: Lula 48,43%; 2026 1T: Flávio 47,03%, Lula 45,16%).

In [ ]:
nacional = (
    votos.filter(pl.col("papel").is_in(["pt", "adversario", "outro"]))
    .group_by("ano", "turno", "nm_votavel")
    .agg(pl.col("votos").sum())
    .with_columns(
        (100 * pl.col("votos") / pl.col("votos").sum().over("ano", "turno"))
        .round(2)
        .alias("% válidos")
    )
    .filter(pl.col("% válidos") >= 1)
    .sort("ano", "turno", "votos", descending=[False, False, True])
)
nacional

Abstenção e fatia do PT entre os dois maiores, **só nos municípios do Brasil** (sem o
exterior). Por isso a abstenção de 2026 aparece como 20,8%, e não os 21,1% oficiais.

In [ ]:
comparecimento = (
    painel.group_by("ano", "turno")
    .agg(pl.col("aptos", "comparecimento", "abstencoes", "votos_pt", "votos_adv").sum())
    .with_columns(
        (100 * pl.col("abstencoes") / pl.col("aptos")).round(2).alias("% abstenção"),
        (100 * pl.col("votos_pt") / (pl.col("votos_pt") + pl.col("votos_adv")))
        .round(2)
        .alias("PT % 2 maiores"),
    )
    .sort("ano", "turno")
)
comparecimento

## 2. Como o voto no PT se distribui entre os municípios

**Como ler:** cada linha é um ano (1º turno). No eixo x, a fatia do PT entre os dois
maiores; no eixo y, quantos municípios caem em cada faixa de 2,5 pp. A linha tracejada
marca 50%: à esquerda, o adversário venceu no município; à direita, o PT.

In [ ]:
largura_faixa = 2.5
hist = (
    painel.filter(pl.col("turno") == 1)
    .with_columns(
        ano=pl.col("ano").cast(pl.String),
        faixa=(100 * pl.col("pt_2p") / largura_faixa).floor() * largura_faixa,
    )
    .group_by("ano", "faixa")
    .agg(pl.len().alias("municipios"))
    .sort("ano", "faixa")
)

In [ ]:
linhas = (
    alt.Chart(hist)
    .mark_line(interpolate="step-after")
    .encode(
        x=alt.X("faixa:Q", title="PT entre os dois maiores (%)", scale=alt.Scale(domain=[0, 100])),
        y=alt.Y("municipios:Q", title="Municípios"),
        color=alt.Color(
            "ano:N", scale=alt.Scale(domain=["2022", "2026"], range=SERIES[:2]), title="1º turno"
        ),
        tooltip=[
            alt.Tooltip("ano:N", title="Ano"),
            alt.Tooltip("faixa:Q", title="Faixa a partir de (%)"),
            alt.Tooltip("municipios:Q", title="Municípios"),
        ],
    )
)
meio = alt.Chart().mark_rule(color=TINTA["apagada"], strokeDash=[4, 4]).encode(x=alt.datum(50))
(linhas + meio).properties(
    width=640, height=260, title="Distribuição dos municípios por voto no PT (1º turno)"
)

**O que observar:** em 2022 a distribuição tem dois picos, os redutos do adversário
(~40%) e os do PT (~80%). Em 2026 a curva inteira se desloca para a esquerda, e o pico dos
redutos do PT desce mais (de ~80% para ~70%) do que o do adversário. A queda não foi
uniforme entre os municípios.

## 3. A queda de Lula e o tamanho do município

Agora a variação de `pt_2p` entre o 1º turno de 2022 e o de 2026, em pp, para cada AMC.
**Negativo = o PT perdeu espaço para o adversário.** O tamanho é o eleitorado apto em 2026.

In [ ]:
uf_regiao = dict(municipios.select("uf", "regiao").unique().iter_rows())
nomes_amc = municipios.group_by("amc_id").agg(pl.col("nome").sort().str.join(" + "))
delta = (
    amc.filter(pl.col("turno") == 1)
    .pivot(on="ano", index=["amc_id", "uf"], values=["pt_2p", "aptos"])
    .join(nomes_amc, on="amc_id")
    .with_columns(
        delta_pp=100 * (pl.col("pt_2p_2026") - pl.col("pt_2p_2022")),
        pt_2022=100 * pl.col("pt_2p_2022"),
        regiao=pl.col("uf").replace_strict(uf_regiao),
    )
)

**Primeiro, os números de referência.** A variação nacional pesa cada município pelo seu
eleitorado; a mediana dos municípios trata todos igualmente. Se a mediana cai mais que o
total nacional, os municípios pequenos variaram mais que os grandes.

In [ ]:
pt_nacional = comparecimento.filter(pl.col("turno") == 1)["PT % 2 maiores"]
print(f"Variação nacional (pp):            {pt_nacional[1] - pt_nacional[0]:+.2f}")
print(f"Mediana dos municípios (pp):       {delta['delta_pp'].median():+.2f}")
print(f"Municípios onde o PT subiu:        {(delta['delta_pp'] > 0).sum()} de {delta.height}")
delta.select("delta_pp").describe()

**Como ler o gráfico abaixo:** um painel por região. Cada ponto é um município; o eixo x
está em escala log (cada marca é 10× a anterior). A linha preta é uma suavização LOESS:
a tendência média da variação conforme o tamanho. A linha cinza horizontal é o zero.
Olhe a inclinação da linha preta, não a nuvem: municípios muito pequenos são ruidosos.

In [ ]:
ordem_regioes = ["N", "NE", "CO", "SE", "S"]
dados_dispersao = delta.select("nome", "uf", "regiao", "aptos_2026", "pt_2022", "delta_pp")
eixo_x = alt.X(
    "aptos_2026:Q",
    title="Eleitores aptos em 2026 (escala log)",
    scale=alt.Scale(type="log"),
    axis=alt.Axis(values=[1e3, 1e4, 1e5, 1e6, 1e7], format="~s"),
)
pontos = (
    alt.Chart()
    .mark_circle(size=12, opacity=0.25, color=SERIES[0])
    .encode(
        x=eixo_x,
        y=alt.Y("delta_pp:Q", title="Variação do PT entre os 2 maiores (pp)"),
        tooltip=[
            alt.Tooltip("nome:N", title="Município"),
            alt.Tooltip("uf:N", title="UF"),
            alt.Tooltip("aptos_2026:Q", title="Aptos 2026", format=",d"),
            alt.Tooltip("pt_2022:Q", title="PT 2022 (%)", format=".1f"),
            alt.Tooltip("delta_pp:Q", title="Variação (pp)", format="+.1f"),
        ],
    )
)
suave = (
    alt.Chart()
    .transform_calculate(log_aptos="log(datum.aptos_2026) / LN10")
    .transform_loess("log_aptos", "delta_pp", bandwidth=0.5)
    .transform_calculate(aptos_2026="pow(10, datum.log_aptos)")
    .mark_line(color=TINTA["primaria"], strokeWidth=2)
    .encode(x=eixo_x, y="delta_pp:Q")
)
zero = alt.Chart().mark_rule(color=TINTA["apagada"]).encode(y=alt.datum(0))
alt.layer(zero, pontos, suave, data=dados_dispersao).properties(width=210, height=220).facet(
    facet=alt.Facet("regiao:N", sort=ordem_regioes, title=None), columns=3
).properties(title="Variação do PT entre os dois maiores, 1º turno 2022 → 2026, por município")

**O que observar:** em todas as regiões a linha sobe da esquerda para a direita (municípios
maiores perderam menos), então o padrão não é só o Nordeste puxando a média. A exceção
parcial é o próprio Nordeste: a linha fica quase plana até ~50 mil eleitores e só depois
sobe.

### Média por faixa de tamanho × região

O mesmo padrão resumido numa tabela de cores: a média da variação em cada combinação de
região e faixa de eleitorado, **ponderada pelo eleitorado** de 2026.

**Como ler:** a cor segue a convenção do projeto, **vermelho = rumo ao PT, azul = rumo ao
adversário, cinza = sem variação** (escala de −10 a +10 pp). Quanto mais escuro o azul,
maior a perda do PT.

In [ ]:
faixas = [5_000, 10_000, 20_000, 50_000, 100_000, 500_000]
rotulos = [
    "< 5 mil",
    "5–10 mil",
    "10–20 mil",
    "20–50 mil",
    "50–100 mil",
    "100–500 mil",
    "≥ 500 mil",
]
por_faixa = (
    delta.with_columns(faixa=pl.col("aptos_2026").bin_intervals(faixas, labels=rotulos))
    .group_by("regiao", "faixa")
    .agg(
        ((pl.col("delta_pp") * pl.col("aptos_2026")).sum() / pl.col("aptos_2026").sum()).alias(
            "delta_medio"
        ),
        pl.len().alias("municipios"),
    )
    .with_columns(pl.col("faixa").cast(pl.String))
)

In [ ]:
calor = alt.Chart(por_faixa).encode(
    x=alt.X("faixa:N", sort=rotulos, title="Eleitores aptos em 2026", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("regiao:N", sort=ordem_regioes, title=None),
)
celulas = calor.mark_rect(stroke="#fcfcfb", strokeWidth=2, cornerRadius=4).encode(
    color=alt.Color("delta_medio:Q", scale=escala_divergente(10), title="Variação média (pp)"),
    tooltip=[
        alt.Tooltip("regiao:N", title="Região"),
        alt.Tooltip("faixa:N", title="Faixa"),
        alt.Tooltip("delta_medio:Q", title="Variação média (pp)", format="+.1f"),
        alt.Tooltip("municipios:Q", title="Municípios"),
    ],
)
rotulos_txt = calor.mark_text(fontSize=11).encode(
    text=alt.Text("delta_medio:Q", format="+.1f"),
    # texto claro nas células escuras (|variação| > 6 pp)
    color=alt.condition(
        "abs(datum.delta_medio) > 6", alt.value("#ffffff"), alt.value(TINTA["primaria"])
    ),
)
(celulas + rotulos_txt).properties(
    width=560,
    height=200,
    title="Variação média do PT (pp) por tamanho e região, 1º turno 2022 → 2026",
)

**O que observar:** quase tudo é azul, o PT perdeu espaço em praticamente todas as
combinações. A perda é maior à esquerda (municípios pequenos) e no Sul e Centro-Oeste:
no Sul, −9,5 pp abaixo de 5 mil eleitores contra −2,0 pp acima de 500 mil. As únicas
células perto do cinza são as capitais grandes do Norte e do Centro-Oeste.

A tabela por trás do gráfico, se quiser os números exatos:

In [ ]:
por_faixa.pivot(on="faixa", index="regiao", values="delta_medio").select("regiao", *rotulos).sort(
    pl.col("regiao").replace_strict({r: i for i, r in enumerate(ordem_regioes)})
)

## Antes de concluir qualquer coisa

- **Composição:** tamanho, região, renda e agro andam juntos; os painéis por região só
  controlam o primeiro corte. Próximo passo (M1a): covariáveis do Censo e efeitos fixos.
- **Teto:** onde o PT tinha 85% há mais espaço para cair; repetir com `pt_2p_logit`.
- **Ruído:** municípios minúsculos espalham os pontos; olhe a linha, não as nuvens.
- **Falácia ecológica:** são municípios, não eleitores. "Municípios pequenos se afastaram
  mais do PT" não implica que eleitores de cidades pequenas mudaram mais de voto.